### Hypothesis Testing 
In this notebook we will utilize statistical learning methods to determine whether or not the input variables (X) are effective in explaing the predictor (Y). To do this, we must compute the standard errors of the coeficent for each input and T-test it to determine if the coeficent is statisticaly significant. Our chosen model for this phase will be K-Nearest Neighbors and Logistic Regression. These simple models outputted the best evalution results out of all the different algorithms that we're fitted and tested. 

In [ ]:
# import data 
import pandas as pd 
from pathlib import Path 

# load directory 
dir = Path.cwd()

# point to the data opath file 
X_train = dir.parent / 'Data' / 'X_train.csv'
Y_train = dir.parent / 'Data'/ 'Y_train.csv'

# make data 1D (data will load with index if not ignored)
Y_train = Y_train['Survived'].values.ravel() 

# load data 
X_train = pd.read_csv(X_train)
Y_train = pd.read_csv(Y_train)

# view result 
print(f'Input Training Set {X_train.head()}')
print(f'Predictor Training Set {Y_train.head()}')


Input Training Set    Unnamed: 0  Pclass       Age  SibSp  Parch      Fare  Embarked_644  \
0           0       1 -1.966518      0      2  1.502199         False   
1           1       3 -0.043103      0      0 -0.801924         False   
2           2       3 -2.197328      1      1 -0.481456         False   
3           3       2  0.495454      1      2  0.409283         False   
4           4       2  1.034010      1      1  0.353956         False   

   Embarked_C  Embarked_Q  Embarked_S  Sex_female  Sex_male  Cabin_encoded  \
0       False       False        True       False      True       0.384292   
1       False       False        True       False      True       0.296842   
2       False       False        True        True     False       0.297779   
3       False       False        True       False      True       0.296842   
4       False       False        True       False      True       0.303215   

   interaction4  interaction5  interaction6  interaction7    Ratio1    Ra

# K-Nearest Neighbors
Important Insight: Since KNN does not have p-values because it is a non-parametric model, we will use sequential feature selection which automatically handles the backward elimintation process by checking accuaracy/precision at each step. 


In [ ]:
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.neighbors import KNeighborsClassifier # model 

# initialze model with the gridsearch adjusted parameters 
knn = KNeighborsClassifier(leaf_size=10, weights='distance')

# backward elimination for KKN 
sfs = SequentialFeatureSelector(knn, 
                                n_features_to_select = 'auto', 
                                direction = 'backward', 
                                scoring = 'precision')

# fit data onto model (makes the dataset 1D)
sfs.fit(X_train, Y_train))

# best features
selected_features = X_train.columns[sfs.get_support()]


c:\Users\jmira\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\jmira\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\jmira\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metri

In [5]:
selected_features

Index(['Pclass', 'Age', 'Parch', 'Fare', 'Embarked_C', 'interaction4',
       'interaction5', 'interaction6', 'interaction7', 'Ratio4'],
      dtype='object')

## Logistic Regression 
Important Insight: Redundancy is prevelant between the interaction variables, causing the model to suffer when predicting the raw column fare which is made up of the engieneered features. Also, removing the redundant categories like male and embarked to allow the model to use one category as the baseline. 

In [ ]:
import statsmodels.api as sm 

#convert bool objects to floats (statsmodel has a hard time seeing them as bool objects)
X_train_numeric = X_train.astype(float)
                    
# manually add a constant intercept to the data 
X_train_const = sm.add_constant(X_train_numeric)

# fit the model
model = sm.Logit(Y_train, X_train_const)
result = model.fit(maxiter = 1000)

# view results 
print(f'Summary {result.summary()}')

         Current function value: 0.434132
         Iterations: 1000
Summary                            Logit Regression Results                           
Dep. Variable:                      y   No. Observations:                  623
Model:                          Logit   Df Residuals:                      605
Method:                           MLE   Df Model:                           17
Date:                Mon, 13 Apr 2026   Pseudo R-squ.:                  0.3416
Time:                        19:54:02   Log-Likelihood:                -270.46
converged:                      False   LL-Null:                       -410.79
Covariance Type:            nonrobust   LLR p-value:                 1.090e-49
                    coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------
const             3.6561        nan        nan        nan         nan         nan
Unnamed: 0       -0.0003      0.001     -0.514

c:\Users\jmira\AppData\Local\Programs\Python\Python313\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


In [27]:
# variance inflation factor
from statsmodels.stats.outliers_influence import variance_inflation_factor

# intercept has to be constant, but we did this already upstream 

# calculate VIF for each feature 
vif_data = pd.DataFrame()
vif_data['feature'] = X_train.columns
vif_data['VIF'] = [variance_inflation_factor(X_train.values, i)]
print(vif_data.sort_values(by = 'VIF', ascending = False))

NameError: name 'i' is not defined

In [ ]:
#